# Genizah v2.0a — new-generation data (mash-fixed GT, 2.7× letters) + grounding/VQA

First run on **`isaacmg/genizah_ktiv_v2`**: 1,587 manuscripts / 3,698 pages rebuilt
with the repaired layout reconstructor (interleave rescue, adaptive merge, served
fallback), 2.19M letters of page GT (2.7× v1.9a's KTIV slice), plus the new task
families — `locate`, `read_box`, `layout_qa`, `grounded_page` (0–1000-normalized
boxes, verified against real images).

## ⬅️ THE DECISION KNOB (cell 3): `MERGER_MODE`
- `"frozen"` — v1.9a config (merger untouched). **Default / safe.**
- `"lora"`   — v1.9b config (r16 LoRA on the 8 merger linears). Not recommended
  (two-sided at r16 per the ablation study).
- `"full"`   — v1.9c config (full-weight merger via `modules_to_save`).
  **Flip to this if the v1.9c step-700 comparison wins.**

Warm start defaults to the v1.9a flagship (step 1300). To continue from a v1.9c
checkpoint instead (only with `MERGER_MODE="full"`), set `WARM_CKPT_REPO` to
`isaacmg/qwen3-vl-8b-hebrew-v19c-ckpt` and `WARM_REVISION` to the commit titled
*"Training in progress, step 700, checkpoint"* on that repo.

## Prerequisites
1. Colab secret `HF_TOKEN` (write scope) and `WANDB_API_KEY`.
2. **The account must be under its private-storage quota** — `genizah_ktiv_v2`,
   `genizah_clean_v2` and `talmud_finetune_v2` are private, and Hugging Face
   blocks private downloads while over quota (403 "storage limit reached").
3. Checkpoints push PUBLICLY to `isaacmg/qwen3-vl-8b-hebrew-v20a-ckpt`.

Schedule: 2000-step cosine @ 5e-5, eval every 100, kill-anytime/resumable.
If `MERGER_MODE="full"`: eval@100 tripwire ≈0.78-expected; > 0.82 ⇒ restart
at `learning_rate=2e-5`.


In [ ]:
# Cell 1 — installs + env (PINNED: the exact triplet audited 2026-08-09;
# unpinned installs float and transformers 5.x would invalidate the vision-path
# analysis AND the 108-tensor count)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]==2026.8.9" "unsloth_zoo==2026.8.6" "transformers==4.57.6" hf_transfer wandb

from importlib.metadata import version
assert version("unsloth_zoo") == "2026.8.6", f"unsloth_zoo drifted: {version('unsloth_zoo')}"
assert version("transformers") == "4.57.6", f"transformers drifted: {version('transformers')}"

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: genizah_ktiv_v2 (transcription + grounding) + clean_v2 + synth3 + talmud replay
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset, concatenate_datasets

login(token=userdata.get("HF_TOKEN"))

KTIV2_REPO = "isaacmg/genizah_ktiv_v2"
KTIV2_REVISION = "1bb20e209a3b3095fc84dad7c928da4310269fd3"  # 30,991 rows, mash-fixed GT
GENIZAH_REPO = "isaacmg/genizah_clean_v2"
GENIZAH_REVISION = "57366ad378946918731ad0012d699acc7d9ed31c"  # 1,055/56 repaired+screened
SYNTH3_REPO = "isaacmg/synthetic_hebrew_v3"
SYNTH3_REVISION = "59abcf7c30fb6753b9df89f0a099b07a68059013"  # 12 faces, probe-weighted drills
TALMUD_REPO = "isaacmg/talmud_finetune_v2"

for _sha in (KTIV2_REVISION, GENIZAH_REVISION, SYNTH3_REVISION):
    assert len(_sha) == 40, f"unpinned revision: {_sha!r}"

ktiv2 = load_dataset(KTIV2_REPO, split="train", revision=KTIV2_REVISION)
ktiv2_val = load_dataset(KTIV2_REPO, split="val", revision=KTIV2_REVISION)
genizah = load_dataset(GENIZAH_REPO, split="train", revision=GENIZAH_REVISION)
genizah_val = load_dataset(GENIZAH_REPO, split="val", revision=GENIZAH_REVISION)
synth3 = load_dataset(SYNTH3_REPO, split="train", revision=SYNTH3_REVISION)
synth3_eval = load_dataset(SYNTH3_REPO, split="eval", revision=SYNTH3_REVISION)
talmud = load_dataset(TALMUD_REPO, split="train")
talmud_val = load_dataset(TALMUD_REPO, split="val")

# v2.0 task families
ktiv_pages = ktiv2.filter(lambda t: t == "fragment_transcribe", input_columns="task")
ktiv_regions = ktiv2.filter(lambda t: t == "region_transcribe", input_columns="task")
ktiv_crops = ktiv2.filter(
    lambda t: t in ("section_transcribe", "line_transcribe"), input_columns="task")
GROUNDING_TASKS = ("locate", "read_box", "layout_qa", "grounded_page")
grounding = ktiv2.filter(lambda t: t in GROUNDING_TASKS, input_columns="task")

# Talmud replay (forgetting protection; same v16-lesson exclusions as v18/v19)
crops = talmud.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = talmud.filter(lambda t: t == "page_extract", input_columns="task")
gemara_crops = crops.filter(lambda s: s == "gemara", input_columns="section")
pages_small = pages.filter(
    lambda s: s in ("rashi", "tosafot"), input_columns="section")
_n_before = len(pages_small)
pages_small = pages_small.filter(
    lambda st: st.split("_")[0] not in ("16", "05"), input_columns="stem")
print(f"pages_small: {_n_before} -> {len(pages_small)} after vol 16/05 exclusion")
assert len(pages_small) < _n_before, "vol 16/05 exclusion filtered nothing"

print(f"ktiv2: pages={len(ktiv_pages)} regions={len(ktiv_regions)} "
      f"crops={len(ktiv_crops)} grounding={len(grounding)} val={len(ktiv2_val)} | "
      f"genizah={len(genizah)} synth3={len(synth3)} "
      f"pages_small={len(pages_small)} gemara_crops={len(gemara_crops)}")
assert len(ktiv_pages) > 3000 and len(ktiv_regions) > 4000 and len(ktiv_crops) > 8000
assert len(grounding) > 12000 and len(ktiv2_val) >= 1200
assert len(genizah) > 1000 and len(genizah_val) >= 50
assert len(synth3) > 10000 and len(synth3_eval) >= 800

# Label hygiene (transcription answers only — grounding answers are JSON/lines)
for _name, _ds in (("genizah", genizah), ("ktiv", ktiv_pages)):
    _sample = _ds.shuffle(seed=0).select(range(200))["answer"]
    assert all("\u2423" not in a for a in _sample), f"{_name}: internal gap token leaked"
    assert all(not any(ch in a for ch in "&#$_{}<>\\") for a in _sample), f"{_name}: mojibake leaked"
    assert any("[...]" in a for a in _sample), f"{_name}: expected damage-gap markers"
_q = ktiv_regions.shuffle(seed=0).select(range(50))["question"]
assert all("Transcribe ONLY" in q for q in _q), "region rows lost their restriction"

# Grounding hygiene: coordinates must be valid 0-1000 JSON; prompts self-describing
import json as _json
_loc = grounding.filter(lambda t: t == "locate", input_columns="task")                 .shuffle(seed=0).select(range(50))
for _r in _loc:
    _b = _json.loads(_r["answer"])["bbox_2d"]
    assert len(_b) == 4 and all(0 <= v <= 1000 for v in _b), f"bad locate box {_b}"
    assert "0-1000" in _r["question"]
_gp = grounding.filter(lambda t: t == "grounded_page", input_columns="task")                .shuffle(seed=0).select(range(20))
for _r in _gp:
    _arr = _json.loads(_r["answer"])
    assert isinstance(_arr, list) and all(
        len(e["bbox_2d"]) == 4 and all(0 <= v <= 1000 for v in e["bbox_2d"])
        and e["text"] for e in _arr), "bad grounded_page payload"
_rb = grounding.filter(lambda t: t == "read_box", input_columns="task")                .shuffle(seed=0).select(range(50))["question"]
assert all("bbox_2d = [" in q and "0-1000" in q for q in _rb)
print("grounding hygiene OK: locate/read_box/grounded_page validated")


In [ ]:
# Cell 3 — model at page resolution + THE MERGER KNOB + flagship warm start
from unsloth import FastVisionModel
from unsloth_zoo.peft_utils import get_peft_regex
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor (train/infer resolution contract, ships in the export).
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

# ⬅️ THE DECISION KNOB — set per the v1.9c verdict:
#   "frozen" = v1.9a config (default, safe)   "full" = v1.9c full-weight merger
#   "lora"   = v1.9b merger LoRA (two-sided at r16 per the ablation; avoid)
MERGER_MODE = "frozen"
assert MERGER_MODE in ("frozen", "lora", "full")

# Warm start: v1.9a flagship step 1300. For a v1.9c warm start use
# WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v19c-ckpt" with the SHA of the
# commit "Training in progress, step 700, checkpoint" — and MERGER_MODE="full".
WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v19a-ckpt"
WARM_REVISION = "43e21bd7a6fedd323879fc5ea2c298df90a92784"  # v1.9a step 1300

TRAIN_VISION_LORA = True    # unchanged since v1.8b
MERGER_MODULES = [
    "visual.merger.linear_fc1", "visual.merger.linear_fc2",
    *[f"visual.deepstack_merger_list.{i}.linear_fc{j}"
      for i in range(3) for j in (1, 2)],
]

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

base_regex = get_peft_regex(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
)
MERGER_REGEX = r".*\.visual\.(?:merger|deepstack_merger_list\.\d+)\.linear_fc[12]"
target_modules = (f"(?:{base_regex})|(?:{MERGER_REGEX})"
                  if MERGER_MODE == "lora" else base_regex)

if MERGER_MODE == "full":
    # modules_to_save cannot train a quantized clone (see v1.9c notebook)
    import bitsandbytes as bnb
    _mods = {n: m for n, m in model.named_modules()
             if any(n.endswith(s) for s in MERGER_MODULES)}
    assert len(_mods) == 8, f"merger modules found: {sorted(_mods)}"
    _quantized = [n for n, m in _mods.items()
                  if isinstance(m, (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt))]
    assert not _quantized, (
        f"merger modules are quantized: {_quantized} — reload with these in "
        "llm_int8_skip_modules or load_in_4bit=False before proceeding.")

model = FastVisionModel.get_peft_model(
    model,
    target_modules=target_modules,
    modules_to_save=MERGER_MODULES if MERGER_MODE == "full" else None,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# creation gate — the run is void unless the adapter matches the KNOB
_tower_lora = [n for n, _ in model.named_parameters()
               if ".visual.blocks." in n and "lora_A" in n]
_merger_lora = [n for n, _ in model.named_parameters()
                if "merger" in n and "lora_A" in n]
_merger_full = [n for n, p in model.named_parameters()
                if "merger" in n and ".modules_to_save." in n
                and n.endswith(".weight") and p.requires_grad]
assert len(_tower_lora) == 108, f"tower adapters: {len(_tower_lora)}/108"
if MERGER_MODE == "frozen":
    assert not _merger_lora and not _merger_full, "merger adapted but knob=frozen"
elif MERGER_MODE == "lora":
    assert len(_merger_lora) == 8 and not _merger_full,         f"merger lora {len(_merger_lora)}/8 — knob=lora not honoured"
else:
    assert len(_merger_full) == 8 and not _merger_lora,         f"merger clones {len(_merger_full)}/8 — knob=full not honoured"
print(f"MERGER_MODE={MERGER_MODE}: tower {len(_tower_lora)}/108, "
      f"merger lora {len(_merger_lora)}, merger full {len(_merger_full)}")

# weights-only warm start (fresh optimizer + schedule)
assert len(WARM_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(WARM_CKPT_REPO, revision=WARM_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
_sd = load_file(f"{local}/last-checkpoint/adapter_model.safetensors")
_loaded_merger_lora = any("merger" in k and "lora" in k for k in _sd)
_loaded_merger_full = any("merger" in k and "lora" not in k for k in _sd)
# (modules_to_save saves PLAIN keys — the adapter-name infix is stripped on save)
# the knob must match what the warm checkpoint carries, or trained merger
# weights would be silently dropped on load
if _loaded_merger_full:
    assert MERGER_MODE == "full", "warm ckpt has full merger weights — set MERGER_MODE='full'"
if _loaded_merger_lora:
    assert MERGER_MODE == "lora", "warm ckpt has merger LoRA — set MERGER_MODE='lora'"
if MERGER_MODE == "full":
    # PEFT's modules_to_save loader KeyErrors when the incoming dict lacks the
    # merger keys (any pre-v1.9c warm start). Inject the base weights — the
    # identity start the gate below verifies; setdefault keeps warm-loaded
    # merger weights when the checkpoint carries them.
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            for _pn, _p in _m.original_module.named_parameters():
                _sd.setdefault(f"{_n}.{_pn}", _p.detach().clone())
missing = set_peft_model_state_dict(model, _sd)
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))
_wv = [n for n, p in model.named_parameters()
       if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
assert len(_wv) == 108, f"warm start vision adapter not loaded: {len(_wv)}/108 nonzero"
if MERGER_MODE == "lora" and not _loaded_merger_lora:
    _mz = [n for n, p in model.named_parameters()
           if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert not _mz, f"merger lora_B nonzero at start: {_mz[:3]}"
if MERGER_MODE == "full":
    _wrappers = [(n, m) for n, m in model.named_modules()
                 if any(n.endswith(s) for s in MERGER_MODULES)
                 and hasattr(m, "modules_to_save")]
    assert len(_wrappers) == 8, f"modules_to_save wrappers: {len(_wrappers)}/8"
    if not _loaded_merger_full:
        for _n, _w in _wrappers:
            assert torch.allclose(
                _w.modules_to_save["default"].weight.detach().float(),
                _w.original_module.weight.detach().float()),                 f"merger clone differs from base at start: {_n}"
        print("merger clones at identity (fresh full-weight start)")
    else:
        print("merger clones warm-loaded from checkpoint")
print(f"warm start OK from {WARM_CKPT_REPO}@{WARM_REVISION[:8]}")

if TRAIN_VISION_LORA:
    # patch-embed require-grad fix (see v1.8b/v1.9 notebooks for the diagnosis)
    def _vision_embeds_require_grad(module, inputs, output):
        if not torch.is_grad_enabled():
            return output
        output.requires_grad_(True)
        return output
    _patch_embed = next(m for n, m in model.named_modules()
                        if n.endswith("visual.patch_embed"))
    _patch_embed.register_forward_hook(_vision_embeds_require_grad)
    print("vision LoRA gradient fix: ON")

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable/1e6:.1f}M")
print("resolution:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch must show page-res pixels + masked labels
batch = collator([ktiv_pages[0], synth3[0]])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
assert ktiv_pages[0]["answer"][:30] in _tok.decode(unmasked)
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked")

In [ ]:
# Cell 5 — gradient-flow verification (cheap; run BEFORE burning GPU-hours)
# Language LoRA always; tower LoRA iff TRAIN_VISION_LORA; merger per the KNOB.
FastVisionModel.for_training(model)
_vb = {k: (v.to(model.device) if torch.is_tensor(v) else v)
       for k, v in collator([ktiv_pages[0]]).items()}
model(**_vb).loss.backward()
tower_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.grad is not None]
mrg_lora_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and "lora_B" in n and p.grad is not None]
mrg_full_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and ".modules_to_save." in n and n.endswith(".weight")
              and p.grad is not None]
lang_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
          if ".visual." not in n and "lora_B" in n and p.grad is not None]
model.zero_grad(set_to_none=True)
assert lang_g and max(lang_g) > 0, "language LoRA got no gradients"
if TRAIN_VISION_LORA:
    assert len(tower_g) == 108 and min(tower_g) > 0, (
        f"tower LoRA dead or partial: {len(tower_g)}/108 tensors with grads")
if MERGER_MODE == "lora":
    assert len(mrg_lora_g) == 8 and min(mrg_lora_g) > 0,         f"merger LoRA dead or partial: {len(mrg_lora_g)}/8"
elif MERGER_MODE == "full":
    assert len(mrg_full_g) == 8 and min(mrg_full_g) > 0,         f"merger full weights dead or partial: {len(mrg_full_g)}/8"
else:
    assert not mrg_lora_g and not mrg_full_g, "merger got grads but knob=frozen"
print(f"grad gate OK (MERGER_MODE={MERGER_MODE}): tower min|g|={min(tower_g):.2e} "
      f"language max|g|={max(lang_g):.2e}")


In [ ]:
# Cell 6a — OPTIONAL throughput benchmark (flag-gated; run once, then set False)
# The A100 ran v18 at batch 1x8 with 13.8/40GB VRAM — headroom. This times
# fwd+bwd for candidate batch geometries at the REAL resolution so the long
# run uses the fastest safe config. To compare gradient-checkpointing modes
# ("unsloth" vs True), change it in cell 3 and rerun cells 3-6a once each.
RUN_THROUGHPUT_BENCH = False
BATCH_GEOMETRIES = [(1, 8), (2, 4), (4, 2)]   # (per_device_batch, grad_accum)

if RUN_THROUGHPUT_BENCH:
    import time
    FastVisionModel.for_training(model)
    bench_rows = [ktiv_pages[i] for i in range(8)] + [synth3[i] for i in range(8)]
    for bs, ga in BATCH_GEOMETRIES:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time(); n_micro = 0
            for step in range(2):                    # 2 optimizer steps
                for micro in range(ga):
                    rows = [bench_rows[(n_micro + i) % len(bench_rows)] for i in range(bs)]
                    b = {k: (v.to(model.device) if torch.is_tensor(v) else v)
                         for k, v in collator(rows).items()}
                    (model(**b).loss / ga).backward()
                    n_micro += 1
                model.zero_grad(set_to_none=True)
            dt = (time.time() - t0) / 2
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"batch {bs}x{ga}: {dt:.1f}s/optimizer-step, peak {peak:.1f}GB")
        except torch.cuda.OutOfMemoryError:
            print(f"batch {bs}x{ga}: OOM — skip")
            torch.cuda.empty_cache()
    model.zero_grad(set_to_none=True)
    print("pick the fastest non-OOM geometry and set it in cell 6, then set "
          "RUN_THROUGHPUT_BENCH = False")

In [ ]:
# Cell 6 — v2.0a mixture (+GROUNDING_SHARE knob) + per-domain eval + training
import inspect
import wandb
from datasets import concatenate_datasets, interleave_datasets
from huggingface_hub import list_repo_files
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v20a-ckpt"  # PUBLIC from the start
OUT_DIR = "outputs_v20a"
wandb.login(key=userdata.get("WANDB_API_KEY"))

# v2.0a mixture: transcription backbone + grounding at a design share of ~8%
# (docs/v20_grounding_qa_design.md targets 5-10%; adjust GROUNDING_SHARE only).
GROUNDING_SHARE = 0.08
_probs = [0.30, 0.19, 0.15, 0.07, 0.07, 0.09, 0.05, GROUNDING_SHARE]
assert abs(sum(_probs) - 1.0) < 1e-9, f"mixture sums to {sum(_probs)}"
mixture = interleave_datasets(
    [ktiv_pages, genizah, synth3, ktiv_regions, ktiv_crops,
     pages_small, gemara_crops, grounding],
    probabilities=_probs, seed=3407,
    stopping_strategy="all_exhausted",
)
# val loss every 100 steps across every domain INCLUDING grounding
eval_ds = concatenate_datasets([
    ktiv2_val.filter(lambda t: t == "fragment_transcribe", input_columns="task")
             .select(range(40)),
    genizah_val.select(range(40)),
    ktiv2_val.filter(lambda t: t == "region_transcribe", input_columns="task")
             .select(range(15)),
    talmud_val.filter(lambda t: t == "page_extract", input_columns="task")
              .select(range(30)),
    talmud_val.filter(
        lambda t, s: t == "crop_transcribe" and s == "gemara",
        input_columns=["task", "section"]).select(range(15)),
    synth3_eval.select(range(20)),
    ktiv2_val.filter(lambda t: t == "locate", input_columns="task").select(range(15)),
    ktiv2_val.filter(lambda t: t == "read_box", input_columns="task").select(range(10)),
    ktiv2_val.filter(lambda t: t == "layout_qa", input_columns="task").select(range(10)),
])

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*",
                          local_dir=OUT_DIR)
        resume_dir = f"{OUT_DIR}/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=mixture, eval_dataset=eval_ds,
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        max_steps=2000,                    # kill-anytime, resumes
        learning_rate=5e-5,                # if MERGER_MODE="full" and
                                           # eval@100 > 0.82: restart at 2e-5
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=False,
        optim="adamw_8bit", seed=3407, output_dir=OUT_DIR,
        report_to="wandb", run_name="genizah_v20a",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
trainer.train(resume_from_checkpoint=resume_dir)


In [ ]:
# Cell 7 — export merged (policy-carrying) model, gated per the KNOB
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-v20a-merged"
if TRAIN_VISION_LORA:
    _nz = [n for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_nz) == 108, f"shipped adapter tower lora_B nonzero: {len(_nz)}/108"
if MERGER_MODE == "lora":
    _mnz = [n for n, p in model.named_parameters()
            if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_mnz) == 8, f"merger lora_B nonzero: {len(_mnz)}/8 — merger never moved"
elif MERGER_MODE == "full":
    _moved = []
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            _c = _m.modules_to_save["default"].weight.detach().float()
            _o = _m.original_module.weight.detach().float()
            if not torch.allclose(_c, _o):
                _moved.append(_n)
    assert len(_moved) == 8, f"merger full weights moved: {len(_moved)}/8"
print(f"shipped checks OK (MERGER_MODE={MERGER_MODE})")
model.save_pretrained_merged("v20a-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v20a-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)
